# Auto-Ingestion: From Database to Datasets

TL;DR: SLayer can introspect a database schema and automatically generate datasets with dimensions, metrics, and relationships matching the schema.  

This notebook shows what happens under the hood.

We use the **Jaffle Shop** dataset — a synthetic e-commerce schema with 7 tables and foreign key relationships between them.

**Prerequisites:** `pip install motley-slayer`

See also: [Auto-Ingestion docs](../../concepts/ingestion.md) | [Datasets docs](../../concepts/datasets.md)


In [ ]:
import os
import sys

import duckdb
import sqlalchemy as sa

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop, DB_PATH

engine, storage, datasets = ensure_jaffle_shop()

## The Jaffle Shop Schema

The database has 7 tables connected by foreign keys:

```
customers  <──  orders  ──>  stores
                  │
              items  ──>  products  <──  supplies

customers  <──  tweets
```

Let's peek at the data:

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)
for table in ["customers", "stores", "products", "orders", "items", "supplies", "tweets"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:>15}: {count:>8,} rows")
conn.close()

## Step 1: FK Discovery

The first thing auto-ingestion does is read the foreign key constraints. Each one means "this table has a FK pointing to that table" and becomes a relationship — even when the FKs form a cycle.

In [ ]:
from slayer.core.models import DatasourceConfig

ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=DB_PATH)
sa_engine = sa.create_engine(ds.resolve_env_vars().get_connection_string())
inspector = sa.inspect(sa_engine)

print("Foreign keys (table -> tables it references):")
for table in sorted(inspector.get_table_names()):
    refs = sorted({fk["referred_table"] for fk in inspector.get_foreign_keys(table)})
    print(f"  {table} -> {refs}" if refs else f"  {table} (no FKs)")

sa_engine.dispose()

Notice that `items` transitively reaches `customers` and `stores` through `orders`, even though it has no direct FK to them. At query time, SLayer walks the relationship graph through `orders` to reach these tables.

## Step 2: Relationship Generation

Each table's own FK relationships become `DatasetRelationship` objects — one per FK. Each relationship specifies:
- **to**: the target dataset
- **from_columns** / **to_columns**: parallel lists of key fields, in pair order — the columns to join on

Only direct FKs are stored. Tables reachable via multiple hops (e.g. `items → orders → customers`) are resolved at query time by walking each intermediate dataset's own relationships.

In [ ]:
orders_dataset = next(d for d in datasets if d.name == "orders")
items_dataset = next(d for d in datasets if d.name == "items")

print("=== orders dataset relationships (direct FKs only) ===")
for r in orders_dataset.relationships:
    src, tgt = r.column_pairs[0]
    print(f"  -> {r.to}  ON {src} = {tgt}")

print()
print("=== items dataset relationships (direct FKs only) ===")
for r in items_dataset.relationships:
    src, tgt = r.column_pairs[0]
    print(f"  -> {r.to}  ON {src} = {tgt}")

print()
print("Note: items has no relationship to customers or stores —")
print("those are reached at query time via orders.relationships.")

## Step 3: Dimension & Metric Generation

For each table, SLayer generates:
- **Dimensions** for every column. See the [relationships](../05_relationships/relationships_nb.ipynb) on how to refer to joined dimensions and metrics. 
- **Metrics**: `count` always; one metric per numeric non-ID column (with `sql` set to the column name); non-numeric non-ID columns also get a metric

In [ ]:
# Iterate the unified `fields` list — v2 merges v1's separate dimensions and
# row-level metrics into one list. Each DatasetField carries data type, primary_key,
# allowed_aggregations (whitelist), and filter (CASE-WHEN at aggregation time).
print("=== orders dataset fields ===")
print(f"{'name':<25} {'sql':<25} {'datatype':<10}")
print("-" * 70)
for fld in orders_dataset.fields:
    pk = " [PK]" if fld.primary_key else ""
    sql_str = fld.sql or fld.name
    print(f"{fld.name:<25} {sql_str:<25} {str(fld.datatype):<10}{pk}")


In [ ]:
# In v2 `dataset.metrics` holds *named expressions* (DatasetMetric objects), not
# row-level metric definitions. After auto-ingest the list is empty by
# default; agents/users populate it via edit_dataset with reusable expressions
# like `{"name": "aov", "expression": "sum(revenue) / count(*)"}`.
print("=== orders dataset metrics (named expressions) ===")
print(f"{len(orders_dataset.metrics)} expression(s) defined.")


Joined dimensions aren't stored on the dataset — they're resolved at query time.

Use dot syntax to walk the relationship graph, including through multiple hops.

In [ ]:
# 1-hop: items -> products
result = engine.execute_sync(
    query={
        "population": "items",
        "metrics": ["count(items.*)"],
        "dimensions": ["products.name"],
        "order": [{"field": "items._count", "direction": "desc"}],
        "limit": 5,
    }
)

print("=== 1-hop: items -> products ===")
for row in result.data:
    print(
        f"  {row['products.name']}: {row['items._count']} items"
    )

# Multi-hop: items -> orders -> customers
result = engine.execute_sync(
    query={
        "population": "items",
        "metrics": ["count(items.*)"],
        "dimensions": ["orders.customers.name"],
        "order": [{"field": "items._count", "direction": "desc"}],
        "limit": 5,
    }
)

print("\n=== Multi-hop: items -> orders -> customers ===")
for row in result.data:
    print(
        f"  {row['orders.customers.name']}: {row['items._count']} items"
    )

## Summary

Auto-ingestion discovers the database structure and generates a complete semantic layer:

1. **Foreign keys** identify table relationships — every FK becomes a relationship, cycles included
2. **Direct relationships** — one `DatasetRelationship` per FK on the source table; multi-hop paths are resolved at query time by walking the graph
3. **Dimensions and metrics** are generated with proper naming conventions

No join-related SQL is baked into the datasets — JOINs are constructed dynamically at query time from each dataset's direct relationship metadata.

**Next:** See the [Queries](../../concepts/queries.md) guide to query this data, or the [Relationships](../05_relationships/relationships_nb.ipynb) notebook for a deep dive into relationship mechanics.